In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tueplots import bundles, cycler, figsizes
from tueplots.constants.color import palettes

plt.rcParams.update(bundles.icml2022())
plt.rcParams.update(figsizes.icml2022_full())
plt.rcParams.update(cycler.cycler(color=palettes.tue_plot))
plt.rcParams.update({"figure.dpi": 350})

# Note that this notebook is highly outdated and doesn't work with the current data!

## Load Data

In [ ]:
path = "../../data/newspaper_collection_evaluation_results_20_12_2025.csv"
result_df = pd.read_csv(path, index_col=False)

## Load Politician Attributes

In [ ]:
original_path = "../../data/politicians/data.csv"
politician_df = pd.read_csv(original_path, index_col=False)
politician_df = politician_df.drop(columns=["Unnamed: 0", "index"])
politician_surnames = politician_df["surname"].unique()

## Add all the other attributes to the politicians

In [ ]:
attributes = ["party", "birth", "gender", "fullname"]
for att in attributes:
    result_df[att] = result_df["surname"].apply(
        lambda surname: politician_df[politician_df["surname"] == surname][att].iloc[0])

In [ ]:
try:
    result_df = result_df.drop(columns=["Unnamed: 0"])
    result_df = result_df.drop(columns=["Unnamed: 0.1"])
except Exception:
    print("Couldn't find the cols")
result_df.info()

In [ ]:
confident_predictions = result_df["confidence"] > 60
print(f"- Number of confident predictions = {confident_predictions.sum()}/{len(result_df)}")
result_df = result_df[confident_predictions]
result_df.columns

In [ ]:
emotions_str = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
emotions_agg_party = result_df[["party"] + emotions_str].groupby("party").mean()
row_sum = result_df[emotions_str].sum(axis=1) # anyway 1
# col_sum = result_df[emotions_str].sum(axis=0)
result_df[emotions_str] = (result_df[emotions_str]) / 100

In [ ]:
agg_plot_data = result_df[["party", "newspaper"] + emotions_str]
for np in result_df["newspaper"].unique().tolist():
    fig, ax = plt.subplots(2, 4, figsize=(8, 3))
    fig.suptitle(f"{np} [{(result_df["newspaper"]==np).sum()}]", fontsize=15)
    for i, party in enumerate(result_df["party"].unique().tolist()):
        row = (i) // 4
        col = i % 4
        # print(row, col)
        sns.barplot(
            agg_plot_data[(agg_plot_data["party"] == party) & (agg_plot_data["newspaper"] == np)], 
            estimator="mean",
            errorbar=("ci", 95), 
            ax=ax[row, col])
        ax[row, col].set_title(party)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f"fig/{np}_{party}.png")

In [ ]:
import numpy as np
# agg_plot_data = result_df[["party", "newspaper"] + emotions_str]
for emotion in emotions_str:
    fig, ax = plt.subplots(2, 4, figsize=(8,3))
    fig.suptitle(emotion, fontsize=15)
    for i, party in enumerate(result_df["party"].unique().tolist()):
        row = (i) // 4
        col = i % 4
        sns.barplot(
            agg_plot_data[(agg_plot_data["party"] == party)][[emotion, "newspaper"]], 
            x="newspaper",
            hue="newspaper",
            y=emotion,
            estimator="mean", 
            ax=ax[row, col])
        ax[row, col].set_yticks(ticks=np.arange(0, 0.5, .1))
        ax[row, col].set_title(party)
    # plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## How are samples distributed?

In [ ]:
## by party
parties = result_df["party"].unique().tolist()
parties_dist = pd.DataFrame({p: [(result_df["party"]==p).sum()] for p in parties})
sns.barplot(
    data=parties_dist,
    alpha=0.9,
    )
plt.title("Distribution of $\\frac{Samples}{Party}$")
plt.show()

In [ ]:
## Average Prediction Confidence By Party
# fig, ax = plt.subplots(1, 1, figsize=(3, 2))
sns.barplot(
    data=result_df,
    y="confidence",
    x="party",
    hue="party",
)
sns.lineplot(
    x=np.arange(-0.5, 7),
    y=result_df["confidence"].mean(),
    linewidth=1,
    linestyle="--",
    label=f"$\\mu$={result_df["confidence"].mean():.3f}",
    )
plt.title("Average Prediction Confidence By Party")
plt.show()


## Columns

- 'name', 
- 'surname', 
- 'confidence', 
- 'distance', 
- 'date', 
- 'article',
- 'newspaper', 
- 'dominant_emotion', 
- 'angry', 
- 'disgust', 
- 'fear', 
- 'happy',
- 'sad', 
- 'surprise', 
- 'neutral', 
- 'party', 
- 'birth', 
- 'gender', 
- 'fullname'


In [ ]:
emotions_str
emotions_dict = {e:i for i, e in enumerate(emotions_str)}
result_df["dominant_emotion"].apply(lambda x: emotions_dict[x])
result_df.columns

In [ ]:
fullnames = result_df["fullname"].value_counts()
ax = sns.barplot(data=fullnames[:15])
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.show()

## Convert to equal time-formats 

In [ ]:
result_df["newspaper"].unique()

In [ ]:
from datetime import datetime

## SZ
sz_df = result_df[result_df["newspaper"] == "sz"]
print("SZ format:", sz_df.iloc[0]["date"])
## Stern
stern_df = result_df[result_df["newspaper"] == "stern"]
print("stern format:", stern_df.iloc[0]["date"])
try:
    ## taz
    taz_df = result_df[result_df["newspaper"] == "taz"]
    print("taz format:", taz_df.iloc[0]["date"])
    ## freitag
    freitag_df = result_df[result_df["newspaper"] == "freitag"]
    print("freitag format:", freitag_df.iloc[0]["date"])
    ## nd
    nd_df = result_df[result_df["newspaper"] == "nd"]
    print("nd format:", nd_df.iloc[0]["date"])
    ## spiegel
    spiegel_df = result_df[result_df["newspaper"] == "spiegel"]
    print("spiegel format:", spiegel_df.iloc[0]["date"])
except IndexError:
    print("...")


"2021-01-31 19:03:46+0100"

In [ ]:
## SZ & Stern have                  : YYYY-MM-DD [...]
## Taz, Freitag, Nd and Spiegel have: DD-MM-YYYY
format_1 = ["taz", "freitag", "nd" ,"spiegel"]
format_2 = ["sz", "stern"]
format_1_idx = result_df["newspaper"].apply(lambda x: x in format_1)
format_2_idx = result_df["newspaper"].apply(lambda x: x in format_2)
print("Entries with Format 1:", format_1_idx.sum())
print("Entries with Format 2:", format_2_idx.sum())

result_df.loc[format_1_idx, "date_conv"] = pd.to_datetime(
    result_df.loc[format_1_idx, "date"],
    dayfirst=True,
    utc=True
)
result_df.loc[format_2_idx, "date_conv"] = pd.to_datetime(
    result_df.loc[format_2_idx, "date"],
    dayfirst=False,
    format="ISO8601",
    utc=True

)
result_df.loc[0, "date_conv"],result_df.loc[1, "date_conv"]

In [ ]:
### SANITY CHECK
single_pol = result_df[result_df["surname"]=="putin"]
sns.scatterplot(
    data=single_pol,
    x=single_pol["date_conv"],
    y=single_pol["dominant_emotion"],
    hue=single_pol["dominant_emotion"]
)
plt.show()

In [ ]:
emotion_grid = pd.crosstab(result_df['newspaper'], result_df['dominant_emotion'])
sns.heatmap(emotion_grid, annot=True, fmt=".2f")
plt.title("Emotional Profile by Newspaper (Raw Frequencies)")
plt.xlabel("Dominant Emotion")
plt.ylabel("Newspaper")
plt.show()
emotion_grid.sum(axis=1)

In [ ]:
emotion_grid = pd.crosstab(result_df['newspaper'], result_df['dominant_emotion'])
emotion_grid_norm = emotion_grid.div(emotion_grid.sum(axis=1), axis=0)
sns.heatmap(emotion_grid_norm, annot=True, fmt=".2f", square=True)
plt.title("Emotional Profile by Newspaper (Normalized Frequencies)")
plt.xlabel("Dominant Emotion")
plt.ylabel("Newspaper")
plt.show()

In [ ]:

result_df["date"] = result_df["date_conv"]
result_df.to_csv(time_line_path)